In [3]:
import os
import cv2
pt = './Dataset1/'

def Frame_extraction(vpt):
    video = cv2.VideoCapture(vpt)
    while True:
        ret, frame = video.read()
        if ret:
            print('No problem...')
            break
        else:
            print('*********Problem...->',vpt)
            break

for i in os.listdir(pt):
    print(i)
    for j in os.listdir(pt+i):
        if j.lower()[0]=='v':
            for k in os.listdir(pt+i+'/'+j):
                print(k)
                Acl = k.split('.')[0]
                Frame_extraction(pt+i+'/'+j+'/'+k)
                
    

Ganesh Babu M
Angry.mp4
No problem...
Disgust.mp4
No problem...
Happy.mp4
No problem...
Neutral.mp4
No problem...
Sad.mp4
No problem...
Surprise.mp4
No problem...
Ghanashree
Angry.mp4
No problem...
Disgust.mp4
No problem...
Happy.mp4
No problem...
Neutral.mp4
No problem...
Sad.mp4
No problem...
Surprise.mp4
No problem...
Gowthami
Angry.mp4
No problem...
Disgust.mp4
No problem...
Happy.mp4
No problem...
Neutral.mp4
No problem...
Sad.mp4
No problem...
Surprise.mp4
No problem...
Gurumurthy
Angry.mp4
No problem...
Disgust.mp4
No problem...
Happy.mp4
No problem...
Neutral.mp4
No problem...
Sad.mp4
No problem...
Surprise.mp4
No problem...
Harini M
Angry.mp4
No problem...
Disgust.mp4
No problem...
Happy.mp4
No problem...
Neutral.mp4
No problem...
Sad.mp4
No problem...
Surprise.mp4
No problem...
Janhvi
Angry.mp4
No problem...
Disgust.mp4
No problem...
Happy.mp4
No problem...
Neutral.mp4
No problem...
Sad.mp4
No problem...
Surprise.mp4
No problem...
Jeevan
Angry.mp4
No problem...
Disgust.mp4
No

In [1]:
import os
import sys
import numpy as np
import cv2
import torch
import torch.backends.cudnn as cudnn
from models.common import DetectMultiBackend
from utils.general import (LOGGER, check_file, check_img_size, check_imshow, check_requirements, colorstr,
                           increment_path, non_max_suppression, print_args, scale_coords, strip_optimizer, xyxy2xywh)
from utils.plots import Annotator, colors, save_one_box1
from utils.torch_utils import select_device, time_sync
from utils.augmentations import Albumentations, augment_hsv, copy_paste, letterbox, mixup, random_perspective
imgsz=640
conf_thres=0.75
iou_thres=0.45
max_det=100
device='cpu'
agnostic_nms=False
line_thickness=3
imgsz = (640,640)
device = select_device(device)
IMG_SIZE = 50

# face model
f_model = DetectMultiBackend('./best.pt', device=device, dnn=False)
f_model.model.float()
f_model(torch.zeros(1, 3, *imgsz).to(device).type_as(next(f_model.model.parameters())))

YOLOv5  2021-11-22 torch 2.2.0+cpu CPU

Fusing layers... 
Model Summary: 213 layers, 7012822 parameters, 0 gradients, 15.8 GFLOPs


tensor([[[4.59752e+00, 5.45897e+00, 1.03397e+01, 1.95258e+01, 3.77059e-07, 9.99999e-01],
         [1.29912e+01, 6.32013e+00, 1.76362e+01, 1.89744e+01, 3.70531e-07, 9.99999e-01],
         [1.83658e+01, 6.96401e+00, 2.58044e+01, 2.35927e+01, 7.66410e-07, 9.99999e-01],
         ...,
         [5.62480e+02, 6.04601e+02, 1.60138e+02, 8.06532e+01, 1.12460e-05, 9.99998e-01],
         [5.85641e+02, 6.04723e+02, 1.18968e+02, 8.27631e+01, 1.23382e-06, 9.99998e-01],
         [6.05794e+02, 6.06700e+02, 8.24730e+01, 8.12496e+01, 2.10837e-07, 9.99998e-01]]])

In [2]:
import shutil
import pandas as pd
import PIL
import mediapipe as mp

def Frame_extraction(vpt):
    video = cv2.VideoCapture(vpt)
    fms=[]
    while True:
        ret, frame = video.read()
        if not ret:
            print('Finished.. Total: ',len(fms))
            break
        else:
            fms.append(frame)
    video.release()
    return fms
    
def read_img(im):
    img = letterbox(im, 640, stride=32, auto=True)[0]
    img = img.transpose((2, 0, 1))[::-1]  
    img = np.ascontiguousarray(img)
    return img, im
    
def Face(Im):
    cpn=read_img(Im)
    im1 = torch.from_numpy(cpn[0]).to(device)        
    im1 = im1.float()  
    im1 /= 255  
    if len(im1.shape) == 3:
        im1 = im1[None] 
    pred1 = f_model(im1, augment=False, visualize=False)
    pred1 = non_max_suppression(pred1, 0.70, iou_thres, None, agnostic_nms, max_det=max_det)
    fc, bbx = [], []
    for j, det1 in enumerate(pred1):
        im01 = cpn[1].copy()
        imc1 = im01.copy() 
        a1 = Annotator(im01, line_width=line_thickness)
        gn1 = torch.tensor(im01.shape)[[1, 0, 1, 0]]
        if len(det1):
            det1[:, :4] = scale_coords(im1.shape[2:], det1[:, :4], im01.shape).round() 
            for *xyxy, conf, cls in reversed(det1):
                obj,sq=save_one_box1(xyxy, imc1, BGR=True)
                fc.append(obj)
                bbx.append(xyxy)
    return fc,bbx

face_mesh = mp.solutions.face_mesh.FaceMesh(static_image_mode=False, max_num_faces=1, min_detection_confidence=0.5)
def face_angle(x,y,w,h):
    yc = int(0.5* h)
    v=y-int(0.5* h)
    cx = int(0.5*w)
    u=(cx-int(0.25* w))/45
    return round((x-cx)/u),round((y-yc)/u)
def Angle(IM):
    h, w, _ = IM.shape
    L = face_mesh.process(cv2.cvtColor(IM, cv2.COLOR_BGR2RGB)).multi_face_landmarks
    if L is None:
        return None,None
    else:
        LM=L[0].landmark
        x1,y1=int(LM[4].x*w),int(LM[4].y*w)
    return face_angle(x1,y1,w,h)

In [3]:
import cv2
from skimage.feature import hog
import tensorflow as tf
from tensorflow.keras.applications.efficientnet import preprocess_input as prp2
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import load_img,img_to_array 
# Load the pre-trained EfficientNet-B0 model
efficientnet=tf.keras.applications.EfficientNetB0(include_top=True, weights="imagenet", input_tensor=None, input_shape=None, pooling=None, classes=1000, classifier_activation="softmax")
efficientnet = Model(inputs=efficientnet.inputs,outputs=efficientnet.layers[-1].output)

def HOG(image):
    image = cv2.resize(image,(256,256))
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    return hog(gray, orientations=8, pixels_per_cell=(32, 32),cells_per_block=(2, 2), visualize=False).tolist()

def EfficientNet(image):
    cv2.imwrite('crop1.jpg',image)
    img=load_img('crop1.jpg',target_size=(224,224))
    img=img_to_array(img)
    img=img.reshape(1,img.shape[0],img.shape[1],img.shape[2])
    img=prp2(img)
    return to_logits(efficientnet.predict(img,verbose=0))[0].numpy().tolist()

def Features(im):
    x,y=Angle(im)
    return EfficientNet(im)+HOG(im)+[x,y]

def clamp_probs(probs):
    eps = torch.finfo(probs.dtype).eps
    return probs.clamp(min=eps, max=1 - eps)

def probs_to_logits(probs, is_binary=False):
    ps_clamped = clamp_probs(probs)
    if is_binary:
        return torch.log(ps_clamped) - torch.log1p(-ps_clamped)
    return torch.log(ps_clamped)

def to_logits(xc):
    return probs_to_logits(torch.tensor(xc, dtype=torch.float32))

In [4]:
import pandas as pd
from sklearn import preprocessing
from sklearn.multiclass import OneVsRestClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC 
scaler = preprocessing.MinMaxScaler()
D1 = pd.read_csv('EfficientNetb0_HOG_pose_FM.csv')
X, Y = D1.iloc[:,:-2].values,D1.iloc[:,-2]
Xo = scaler.fit_transform(X)
lda = LinearDiscriminantAnalysis(n_components=5).fit((Xo),Y)
rmodel =OneVsRestClassifier(SVC(kernel="rbf")).fit(lda.transform(Xo), Y)
def Emotion_predict(fv):
    return rmodel.predict(lda.transform(scaler.transform([fv])))[0]

In [5]:
import os
from Duplicate_Reduction import Duplicates_Removal
import progressbar

def Mood_Pred(ivpt,sp,Acl,nm):
    os.makedirs(sp,exist_ok=True)
    fms=Duplicates_Removal(Frame_extraction(ivpt),0.95).sel
    lb = []
    bar = progressbar.ProgressBar(maxval=len(fms)).start()
    for c,frame in enumerate(fms):
        frame = cv2.resize(frame,(640,640))
        a,b = Face(frame)
        for x,y in zip(a,b):
            if 0 not in x.shape:
                fv = Features(x)
                if None not in fv:
                    label = Emotion_predict(fv)
                    lb.append([c,Acl,label])
        bar.update(c)
    pd.DataFrame(lb,columns = ['Frame_No','Actual','Predicted']).to_csv(sp+'{}_{}_Keyframe_Result.csv'.format(nm,Acl),index=False)

In [6]:
import os
pt = './Data/'
sp1 = './Results/Frames/'
sp2 = './Results/KeyFrames/'
l = []
for i in os.listdir(pt):
    print(i)
    for j in os.listdir(pt+i):
        if j.lower()[0]=='v':
            for k in os.listdir(pt+i+'/'+j):
                print(k)
                Acl = k.split('.')[0]
                Mood_Pred(pt+i+'/'+j+'/'+k,sp2,Acl,i)
                print(Acl," Completed...")
    print(i," child Completed...")
    

Ghanashree
Angry.mp4
Finished.. Total:  2078


  0% |                                                                        |

Taken 2078 Samples reduced to 413 samples.!


 99% |####################################################################### |

Angry  Completed...
Disgust.mp4
Finished.. Total:  1792


...

Sad  Completed...
Surprise.mp4
Finished.. Total:  460


  0% |                                                                        |

Taken 460 Samples reduced to 190 samples.!


 98% |####################################################################### |

Surprise  Completed...
Sanketh  child Completed...


In [7]:
import os
pt = './Dataset1/'
sp1 = './Results/Frames/'
sp2 = './Results/KeyFrames/'
l = []
for i in os.listdir(pt):
    print(i)
    for j in os.listdir(pt+i):
        if j.lower()[0]=='v':
            for k in os.listdir(pt+i+'/'+j):
                print(k)
                Acl = k.split('.')[0]
                Mood_Pred(pt+i+'/'+j+'/'+k,sp2,Acl,i)
                print(Acl," Completed...")
    print(i," child Completed...")

Saini Prakash
Angry.mp4
Finished.. Total:  1140


  0% |                                                                        |

Taken 1140 Samples reduced to 396 samples.!


 98% |####################################################################### |

Angry  Completed...
Disgust.mp4
Finished.. Total:  1403


...

Sad  Completed...
Surprise.mp4
Finished.. Total:  1219


  0% |                                                                        |

Taken 1219 Samples reduced to 313 samples.!


 99% |####################################################################### |

Surprise  Completed...
Yuktha  child Completed...


In [ ]:
import os
from Duplicate_Reduction import Duplicates_Removal

def Mood_Prediction2(ivpt,sp,Acl,nm):
    os.makedirs(sp,exist_ok=True)
    cap = cv2.VideoCapture(ivpt)
    c = 0
    lb = []
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        frame = cv2.resize(frame,(640,640))
        a,b = Face(frame)
        annotator = Annotator(frame, line_width=2)
        for x,y in zip(a,b):
            if 0 not in x.shape:
                cv2.imshow("Face Detection", x)
                fv = Features(x)
                if None not in fv:
                    label = Emotion_predict(fv)
                    lb.append([c,Acl,label])
                else:
                    label = ''
                annotator.box_label(y,label,color=(0, 255, 0), txt_color=(0, 0, 255))
        cv2.imshow("Mood Prediction", annotator.result())
        c +=1
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
    pd.DataFrame(lb,columns = ['Frame_No','Actual','Predicted']).to_csv(sp+'{}_{}_Frames_Result.csv'.format(nm,Acl),index=False)
    cap.release()
    cv2.destroyAllWindows()

def Mood_Prediction1(ivpt,sp,Acl,nm):
    os.makedirs(sp,exist_ok=True)
    fms=Duplicates_Removal(Frame_extraction(ivpt),0.95).sel
    lb = []
    for c,frame in enumerate(fms):
        frame = cv2.resize(frame,(640,640))
        a,b = Face(frame)
        annotator = Annotator(frame, line_width=2)
        for x,y in zip(a,b):
            if 0 not in x.shape:
                #cv2.imshow("Face Detection", x)
                fv = Features(x)
                if None not in fv:
                    label = Emotion_predict(fv)
                    lb.append([c,Acl,label])
                else:
                    label = ''
                annotator.box_label(y,label,color=(0, 255, 0), txt_color=(0, 0, 255))
        #cv2.imshow("Mood Prediction", annotator.result())
    pd.DataFrame(lb,columns = ['Frame_No','Actual','Predicted']).to_csv(sp+'{}_{}_Keyframe_Result.csv'.format(nm,Acl),index=False)
    cv2.destroyAllWindows()

In [ ]:
import os
pt = './Dataset1/'
sp1 = './Results/Frames/'
sp2 = './Results/KeyFrames/'
l = []
for i in os.listdir(pt):
    print(i)
    for j in os.listdir(pt+i):
        if j.lower()[0]=='v':
            for k in os.listdir(pt+i+'/'+j):
                print(k)
                Acl = k.split('.')[0]
                Mood_Prediction1(pt+i+'/'+j+'/'+k,sp2,Acl,i)
                #Mood_Prediction2(pt+i+'/'+j+'/'+k,sp1,Acl,i)
                print(Acl," Completed...")
    print(i," child Completed...")
    

Ganesh Babu M
Angry.mp4
Finished.. Total:  1240
Taken 1240 Samples reduced to 313 samples.!


In [ ]:
import os
pt = 'G:/Children Mood Prediction/Dataset/'
l = []
for i in os.listdir(pt):
    for j in os.listdir(pt+i):
        if j.lower()[0]=='v':
            for k in os.listdir(pt+i+'/'+j):
                ll = k.split('.')[0]
                l.append(ll)
                if ll == 'normal':
                    os.rename(pt+i+'/'+j+'/'+k, pt+i+'/'+j+'/Neutral.mp4')
                
                
    

In [ ]:
import pandas as pd
D1 = pd.read_csv('EfficientNetb0_HOG_pose_FM.csv')
X, Y = D1.iloc[:,:-2].values,D1.iloc[:,-2]
S = list(set(Y))

In [ ]:
S

In [ ]:
set(l)

In [ ]:
import os
from Duplicate_Reduction import Duplicates_Removal
pt='D:/CDATA/'
Ipt='D:/CDATA-out/Image/'
Fpt='D:/CDATA-out/FMs/'
Cpt='D:/CDATA-out/RoI/'

for i in [6]:
    for j in os.listdir(pt+str(i))[-30:]:
        print(i,'standard',j,'Started....')
        c=0
        os.makedirs(Ipt+j,exist_ok=True)
        os.makedirs(Fpt+j,exist_ok=True)
        os.makedirs(Cpt+j,exist_ok=True)
        sp=Cpt+j+'/'
        spp=Ipt+j+'/'
        fv1=[]
        fv2=[]
        lb=[]
        for k in os.listdir(pt+str(i)+'/'+j):
            for l in os.listdir(pt+str(i)+'/'+j+'/'+k):
                ex=l.split('.')[1]
                vp=pt+str(i)+'/'+j+'/'+k+'/'+l
                if ex=='JPG' or ex=='jpg':
                    im=cv2.imread(vp)
                    m=Face(im)
                    if m is not None and len(m[0])!=0:
                        th=cv2.Laplacian(m, cv2.CV_64F).var()
                        im1 = PIL.Image.fromarray(np.uint8(cv2.cvtColor(m, cv2.COLOR_RGB2BGR)))
                        im2 = PIL.Image.fromarray(np.uint8(cv2.cvtColor(im, cv2.COLOR_RGB2BGR)))
                        x,y=Angle(m)
                        if th>30 and x is not None:
                            imnm='{}_{}.jpg'.format(j,str(c))
                            im1.save(sp+imnm)
                            im2.save(spp+imnm)
                            a1,a2=Features(m)
                            fv1.append(a1)
                            fv2.append(a2)
                            lb.append([x,y,j,imnm])
                            c=c+1
                else:
                    fms=Duplicates_Removal(Frame_extraction(vp),0.95).sel
                    for f,m in enumerate(fms):
                        im=Face(m)
                        if im is not None and len(im[0])!=0:
                            th=cv2.Laplacian(im, cv2.CV_64F).var()
                            im1 = PIL.Image.fromarray(np.uint8(cv2.cvtColor(im, cv2.COLOR_RGB2BGR)))
                            im2 = PIL.Image.fromarray(np.uint8(cv2.cvtColor(m, cv2.COLOR_RGB2BGR)))
                            x,y=Angle(im)
                            if th>30 and x is not None:
                                imnm='{}_{}.jpg'.format(j,str(c))
                                im1.save(sp+imnm)
                                im2.save(spp+imnm)
                                a1,a2=Features(im)
                                fv1.append(a1)
                                fv2.append(a2)
                                lb.append([x,y,j,imnm])
                                c=c+1
        pd.concat([pd.DataFrame(fv1),pd.DataFrame(lb,columns=['X-direction','Y-direction','Name','Image_Name'])],axis=1).to_csv(Fpt+j+'/{}_MobileFaceNet_FM.csv'.format(j),index=False) 
        pd.concat([pd.DataFrame(fv2),pd.DataFrame(lb,columns=['X-direction','Y-direction','Name','Image_Name'])],axis=1).to_csv(Fpt+j+'/{}_FaceNet_FM.csv'.format(j),index=False) 
        print(i,'Standard',j,' child Completed..')
    

In [ ]:
import pandas as pd

D1 = pd.read_csv('./FM/EfficientNetb0_FM.csv')
D2 = pd.read_csv('./FM/HOG_FM.csv')
D3 = pd.read_csv('./FM/New_FM.csv')

In [ ]:
A = D1.iloc[:,-1]

In [ ]:
B = D3.loc[:,['X-degree','Y-degree','Image_Name']]

In [ ]:
L = []
for a in A.tolist():
    L.append(B[B.iloc[:,-1]==a].values.tolist()[0][:-1])

In [ ]:
len(L)

In [ ]:
C = pd.DataFrame(L,columns=['X-degree','Y-degree'])

In [ ]:
D = pd.concat([D1.iloc[:,:-2],D2.iloc[:,:-2],C,D2.iloc[:,-2:]],axis=1)

In [ ]:
D.columns = list(range(2568))+D.columns.tolist()[-4:]

In [ ]:
D.to_csv('EfficientNetb0_HOG_pose_FM.csv',index=False)